# Global Wheat Head Domain Analysis

This notebook analyzes copy-paste augmentation on the Global Wheat Head dataset using provenance written by the augmentor.

It focuses on two questions:
- Which source domains are selected by score-guided vs random augmentation?
- How often does augmentation paste objects from a different domain than the target image?

Expected input layout for each experiment:
- `results/<experiment>/Step_4_Copy_Paste_Augmentation/augmented_dataset/train/metadata/*.json`
- `results/<experiment>/Step_4_Copy_Paste_Augmentation/augmented_dataset/train/labels/*.txt`

If you have not regenerated the random run with provenance yet, update the `RANDOM_EXPERIMENT_ROOT` variable in Cell 2.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path("/home/khanh/Projects/DifficultyAgri")
RAW_ROOT = PROJECT_ROOT / "datasets" / "global_wheat_head" / "raw"
YOLO_ROOT = PROJECT_ROOT / "datasets" / "global_wheat_head" / "yolo_format" / "global_wheat_head_yolo"
SCORE_GUIDED_EXPERIMENT_ROOT = PROJECT_ROOT / "results" / "13_global_wheat_head_copy_paste_exp"
RANDOM_EXPERIMENT_ROOT = PROJECT_ROOT / "results" / "05_copy_paste_random_exp"

IMAGE_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def build_domain_map(raw_root: Path) -> dict[str, str]:
    domain_map: dict[str, str] = {}
    for domain_dir in sorted([p for p in raw_root.iterdir() if p.is_dir()]):
        if domain_dir.name == "labels_extracted":
            continue
        for image_path in domain_dir.rglob('*'):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                domain_map[image_path.stem] = domain_dir.name
    return domain_map

def load_yolo_boxes(label_path: Path) -> list[tuple[int, float, float, float, float]]:
    if not label_path.exists():
        return []
    boxes: list[tuple[int, float, float, float, float]] = []
    with label_path.open('r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) != 5:
                continue
            cls_id = int(float(parts[0]))
            x_center, y_center, width, height = map(float, parts[1:])
            boxes.append((cls_id, x_center, y_center, width, height))
    return boxes

def load_original_training_distribution(yolo_root: Path, domain_map: dict[str, str]) -> pd.DataFrame:
    train_images = yolo_root / 'train' / 'images'
    train_labels = yolo_root / 'train' / 'labels'
    rows = []
    for image_path in sorted([p for p in train_images.rglob('*') if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]):
        domain = domain_map.get(image_path.stem, 'unknown')
        boxes = load_yolo_boxes(train_labels / f'{image_path.stem}.txt')
        for box in boxes:
            rows.append({
                'image_name': image_path.name,
                'image_stem': image_path.stem,
                'domain': domain,
                'class_id': int(box[0]),
            })
    return pd.DataFrame(rows)

def load_augmented_provenance(experiment_root: Path, domain_map: dict[str, str]) -> pd.DataFrame:
    metadata_dir = experiment_root / 'Step_4_Copy_Paste_Augmentation' / 'augmented_dataset' / 'train' / 'metadata'
    if not metadata_dir.exists():
        raise FileNotFoundError(f'Provenance metadata not found: {metadata_dir}')

    rows = []
    for metadata_path in sorted(metadata_dir.glob('aug_*.json')):
        with metadata_path.open('r', encoding='utf-8') as f:
            payload = json.load(f)

        background_name = payload.get('background_image_name', '')
        target_domain = domain_map.get(Path(background_name).stem, 'unknown')
        condition = 'score-guided' if payload.get('use_score_guidance', False) else 'random'

        selected_objects = payload.get('selected_objects', [])
        pasted_count = int(payload.get('pasted_object_count', len(selected_objects)))
        for obj in selected_objects:
            source_name = obj.get('source_image_name', '')
            source_domain = domain_map.get(Path(source_name).stem, 'unknown')
            rows.append({
                'condition': condition,
                'augmented_image': metadata_path.stem,
                'background_image_name': background_name,
                'target_domain': target_domain,
                'source_image_name': source_name,
                'source_domain': source_domain,
                'source_object_index': int(obj.get('source_object_index', -1)),
                'score': float(obj.get('score', 0.0)),
                'class_id': int(obj.get('class_id', -1)),
                'pasted_count': pasted_count,
            })

    return pd.DataFrame(rows)

def summarize_domain_distribution(df: pd.DataFrame, all_domains: list[str], value_col: str = 'source_domain') -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['domain', 'count', 'fraction'])

    counts = df[value_col].value_counts().reindex(all_domains, fill_value=0)
    out = counts.rename_axis('domain').reset_index(name='count')
    total = float(out['count'].sum())
    out['fraction'] = out['count'] / total if total else 0.0
    return out

def compute_cross_domain_rate(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return pd.DataFrame(columns=['condition', 'augmented_image', 'cross_domain_rate', 'pasted_count'])

    per_image = df.groupby(['condition', 'augmented_image', 'background_image_name', 'target_domain']).apply(
        lambda g: pd.Series({
            'cross_domain_rate': float((g['source_domain'] != g['target_domain']).mean()),
            'pasted_count': int(len(g)),
        })
    ).reset_index()
    return per_image

domain_map = build_domain_map(RAW_ROOT)
original_df = load_original_training_distribution(YOLO_ROOT, domain_map)
score_guided_df = load_augmented_provenance(SCORE_GUIDED_EXPERIMENT_ROOT, domain_map)
random_df = load_augmented_provenance(RANDOM_EXPERIMENT_ROOT, domain_map) if RANDOM_EXPERIMENT_ROOT.exists() else pd.DataFrame()

print(f'Loaded {len(domain_map)} image-to-domain mappings from raw data.')
print(f'Original training object rows: {len(original_df):,}')
print(f'Score-guided selected objects: {len(score_guided_df):,}')
print(f'Random selected objects: {len(random_df):,}')

display(original_df.head())
display(score_guided_df.head())

## Analysis 1: Domain Assignment of Score-Guided Selected Objects

This section compares the domain distribution of selected pasted objects against the natural training distribution.
It also looks at the top-score selected objects to see whether high-score samples concentrate in minority domains.

In [ ]:
all_domains = sorted(set(domain_map.values()))

natural_domain_df = summarize_domain_distribution(original_df, all_domains, value_col='domain')
score_domain_df = summarize_domain_distribution(score_guided_df, all_domains, value_col='source_domain')
random_domain_df = summarize_domain_distribution(random_df, all_domains, value_col='source_domain') if not random_df.empty else pd.DataFrame(columns=['domain', 'count', 'fraction'])

comparison = natural_domain_df.rename(columns={'count': 'natural_count', 'fraction': 'natural_fraction'})
comparison = comparison.merge(
    score_domain_df.rename(columns={'count': 'score_guided_count', 'fraction': 'score_guided_fraction'}),
    on='domain',
    how='left',
)
comparison = comparison.merge(
    random_domain_df.rename(columns={'count': 'random_count', 'fraction': 'random_fraction'}),
    on='domain',
    how='left',
)
comparison = comparison.fillna(0)
comparison['score_guided_overrep'] = np.where(comparison['natural_fraction'] > 0, comparison['score_guided_fraction'] / comparison['natural_fraction'], np.nan)
comparison['random_overrep'] = np.where(comparison['natural_fraction'] > 0, comparison['random_fraction'] / comparison['natural_fraction'], np.nan)

high_score_threshold = score_guided_df['score'].quantile(0.75) if not score_guided_df.empty else np.nan
high_score_df = score_guided_df[score_guided_df['score'] >= high_score_threshold].copy() if not score_guided_df.empty else pd.DataFrame()
high_score_domain_df = summarize_domain_distribution(high_score_df, all_domains, value_col='source_domain') if not high_score_df.empty else pd.DataFrame(columns=['domain', 'count', 'fraction'])
comparison = comparison.merge(
    high_score_domain_df.rename(columns={'count': 'high_score_count', 'fraction': 'high_score_fraction'}),
    on='domain',
    how='left',
)
comparison = comparison.fillna(0)
comparison['high_score_overrep'] = np.where(comparison['natural_fraction'] > 0, comparison['high_score_fraction'] / comparison['natural_fraction'], np.nan)

display(comparison.sort_values('natural_fraction', ascending=False))

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(comparison))
width = 0.25
ax.bar(x - width, comparison['natural_fraction'], width=width, label='Natural', color='#7f8c8d')
ax.bar(x, comparison['score_guided_fraction'], width=width, label='Score-guided', color='#1f77b4')
if not random_df.empty:
    ax.bar(x + width, comparison['random_fraction'], width=width, label='Random', color='#f39c12')
ax.set_xticks(x)
ax.set_xticklabels(comparison['domain'], rotation=45, ha='right')
ax.set_ylabel('Fraction of objects')
ax.set_title('Domain distribution of selected objects')
ax.legend()
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(12, 5))
ax.bar(comparison['domain'], comparison['score_guided_overrep'], color='#1f77b4')
ax.axhline(1.0, color='black', linestyle='--', linewidth=1)
ax.set_ylabel('Selected fraction / natural fraction')
ax.set_title('Score-guided over-representation ratio by domain')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.show()

print(f'High-score threshold used for top quartile: {high_score_threshold:.4f}')
display(high_score_df[['source_domain', 'score']].sort_values('score', ascending=False).head(20))

## Analysis 2: Cross-Domain Paste Rate

For each augmented image, this computes the fraction of pasted objects whose source domain differs from the target image domain.
A higher rate in score-guided selection would directly support the hypothesis that score guidance increases cross-domain mixing.

In [ ]:
cross_domain_df = compute_cross_domain_rate(pd.concat([score_guided_df, random_df], ignore_index=True)) if not random_df.empty else compute_cross_domain_rate(score_guided_df)

if cross_domain_df.empty:
    print('No provenance rows found. Regenerate the augmented dataset with metadata sidecars enabled.')
else:
    condition_summary = (
        cross_domain_df.groupby('condition')['cross_domain_rate']
        .agg(['count', 'mean', 'std', 'median'])
        .reset_index()
        .rename(columns={'count': 'num_images'})
    )
    display(condition_summary)

    fig, ax = plt.subplots(figsize=(8, 4))
    for idx, condition in enumerate(sorted(cross_domain_df['condition'].unique())):
        values = cross_domain_df.loc[cross_domain_df['condition'] == condition, 'cross_domain_rate'].to_numpy()
        jitter = (np.random.random(size=len(values)) - 0.5) * 0.12
        ax.scatter(np.full_like(values, idx, dtype=float) + jitter, values, alpha=0.7, s=20, label=condition)
    ax.set_xticks(range(len(sorted(cross_domain_df['condition'].unique()))))
    ax.set_xticklabels(sorted(cross_domain_df['condition'].unique()))
    ax.set_ylabel('Cross-domain paste rate per image')
    ax.set_title('Cross-domain paste rate: random vs score-guided')
    plt.tight_layout()
    plt.show()

    print('Interpretation guide: values above 0.5 mean most pasted objects come from other domains than the target image.')
    display(cross_domain_df.sort_values(['condition', 'cross_domain_rate'], ascending=[True, False]).head(20))

## Notes

If `RANDOM_EXPERIMENT_ROOT` does not exist yet, generate a random copy-paste run with the same config except `use_score_guidance: false`, then update Cell 2 and rerun.

The notebook assumes the augmentor writes per-image provenance JSON files under `train/metadata`. That metadata is what makes the domain-level analysis exact.